# CS180 Project 5 - Complete Training Notebook

This notebook runs the tested implementation in `Project5/` on a hosted Google Colab GPU. Before running it in VS Code, upload `Project5_source.zip` to the Colab server; the setup cell extracts it to `/content/Project5`.

In [ ]:
from pathlib import Path
import zipfile
archive = Path('/content/Project5_source.zip')
project_dir = Path('/content/Project5')
if archive.exists() and not project_dir.exists():
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall('/content')
assert project_dir.exists(), 'Upload Project5_source.zip to the Colab server first.'
%cd /content/Project5
!pip install -q -e . pytest
import torch
assert torch.cuda.is_available(), 'Select a GPU Colab runtime before training.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

## Download official data and verify the implementation

In [ ]:
!python download_data.py
!pytest -q
!python visualize_rays.py

## Part 1 - official cat image and hyperparameter tuning

In [ ]:
!python train_2d.py --image data/part1/official_cat.jpg --output-dir outputs/part1/cat --iterations 2000 --batch-size 10000 --sweep --sweep-iterations 1000

## Part 1 - second image from the repository collection

In [ ]:
!python train_2d.py --image data/part1/personal_photo.png --output-dir outputs/part1/personal --iterations 2000 --batch-size 10000

## Part 2 - train NeRF

This keeps the assignment defaults (10K rays, 64 samples, Adam at 5e-4) but trains to 3000 iterations for a cleaner result. If a checkpoint already exists, the cell resumes it instead of throwing away completed training. Microbatches keep GPU memory bounded.

In [ ]:
checkpoint = Path('outputs/part2/checkpoint_latest.pt')
resume = f'--resume {checkpoint}' if checkpoint.exists() else ''
print('Continuing the existing run.' if resume else 'Starting a new run.')
!python train_nerf.py --iterations 3000 --batch-size 10000 --ray-microbatch 1024 --samples 64 --learning-rate 5e-4 --eval-every 250 --checkpoint-every 250 {resume}

## Inspect the final NeRF result

In [ ]:
from IPython.display import Image, display
validation_images = sorted(Path('outputs/part2/validation').glob('step_*.png'))
assert validation_images, 'No validation render was produced.'
print('Latest validation comparison:', validation_images[-1])
display(Image(filename=str(validation_images[-1])))
display(Image(filename='outputs/part2/psnr_curves.png'))
display(Image(filename='outputs/part2/lego_novel_views.gif'))

## Generate the final report after training finishes

In [ ]:
!python generate_report.py
print('Done. Download /content/Project5/outputs to keep all results and the checkpoint.')